# Thermal-Ops with GRPO using TRL (AMD ROCm)

Train an agentic model to manage Data Center Cooling on **AMD GPUs** via PyTorch ROCm (Windows-first). For NVIDIA/CUDA, use `nvidia.ipynb`.


## Prerequisites (read before running)

**Windows 11 (recommended path)**  
- Install **AMD Software: Adrenalin Edition** (e.g. 26.1.1 or newer) and enable **AMD Software: PyTorch on Windows** (ROCm) from the installer, or use the **AI Bundle** if you use that workflow.  
- Use the **Python environment / Jupyter kernel** that ships with or matches AMD's PyTorch on Windows build. Do **not** `pip install` the default CUDA PyTorch wheel from PyPI into that environment.  
- Confirm your GPU appears on AMD's compatibility list for PyTorch on Windows: [PyTorch on Windows Edition 7.2 release notes](https://www.amd.com/en/resources/support-articles/release-notes/RN-AMDGPU-WINDOWS-PYTORCH-7-2.html).  
- On ROCm builds, PyTorch still uses the `torch.cuda` API surface for device selection and availability checks. That is expected on AMD.

**Linux**  
- Install ROCm and PyTorch for ROCm using AMD's instructions; pick matching **manylinux** wheels from [repo.radeon.com/rocm/manylinux](https://repo.radeon.com/rocm/manylinux/) for your Python and ROCm version.  
- If you need real distributed training or FSDP, Linux ROCm is the safer path than Windows ROCm.

**4-bit / bitsandbytes**  
- This notebook trains in FP32 and does not require bitsandbytes.  
- If you add QLoRA or 4-bit later, do **not** assume the standard CUDA wheel will work. Windows ROCm support is still unreliable; prefer current ROCm-specific guidance on Linux.

**TRL GRPO needs full `torch.distributed` (c10d)**  
- **TRL 0.18+** imports `torch.distributed.fsdp` at import time, which requires **`torch._C._distributed_c10d`**.  
- Some Windows PyTorch builds omit c10d entirely, and Windows ROCm wheels may still expose the GPU while lacking those distributed pieces. Verify your exact wheel in the setup cell below.  
- If your wheel lacks c10d, this notebook uses a small import-time stub only for **single-GPU** GRPO with `use_vllm=False`. That fallback is not a real FSDP or multi-GPU solution.  
- If you need a fully supported distributed stack, use Linux ROCm or a Windows build that ships c10d. A separate CUDA wheel environment can restore imports, but it is for NVIDIA CUDA, not AMD GPU training.


In [1]:
# ROCm PyTorch must already be available in this kernel (AMD PyTorch on Windows or Linux ROCm).
%pip install -U trackio ipywidgets datasets

print("Base notebook packages installed.")

Note: you may need to restart the kernel to use updated packages.
Base notebook packages installed.


In [2]:
import torch

print("torch:", torch.__version__)
hip = getattr(torch.version, "hip", None)
if hip:
    print("HIP (ROCm):", hip)
print("GPU via PyTorch CUDA-compatible API:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device 0:", torch.cuda.get_device_name(0))

torch: 2.9.1+rocmsdk20260116
HIP (ROCm): 7.2.26024-f6f897bd3d
GPU via PyTorch CUDA-compatible API: True
Device 0: AMD Radeon RX 9060 XT


### Log in to Hugging Face

Use a token to avoid unauthenticated rate limits: [create a token](https://huggingface.co/settings/tokens) (read is enough for downloads). **Colab:** add a secret named `HF_TOKEN`. **Local / CI:** `export HF_TOKEN=...` or set the same in your environment before imports.


In [3]:
import os

from huggingface_hub import login, notebook_login

_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not _token:
    try:
        from google.colab import userdata

        _token = userdata.get("HF_TOKEN")
    except Exception:
        _token = None

if _token:
    login(token=_token, add_to_git_credential=False)
else:
    notebook_login()


## Define the system prompt


In [4]:
prompt = """You are an expert Data Center Facility Manager with deep knowledge of thermodynamics, power management, and optimal cooling strategies.

Follow these rules to play Thermal-Ops:

1. The target is to maintain 3 server racks within 20.0 to 25.0 Celsius
2. You have a limited number of steps to optimize the temperature while minimizing energy consumption
3. After each step, you receive thermal and cost feedback:
   - GREEN (Optimal): Under 25.0C -> Minimal energy cost applied
   - YELLOW (Warning): Over 25.0C -> Throttling penalty applied
   - RED (Critical): Over 27.0C -> Hardware damage penalty applied
4. All actions must respect physical limits (e.g. max 5000 RPM)
5. You cannot use fans that have experienced a hardware failure
6. Use the tools `set_fan_speed`, `adjust_chiller`, `migrate_workload`, and `wait` to make your moves.
"""



## Define the environment
The `ThermalOpsEnv` class implements all the thermodynamics simulation explicitly inside the environment.


In [5]:
import json
import torch


class ThermalOpsEnv:
    def __init__(self):
        self.num_racks = 3
        self.max_steps = 10
        self.w1_energy = 0.5
        self.w2_overheat = 1000.0
        
        self.safe_temp_max = 25.0
        self.critical_temp = 27.0
        
        self.reset()
        
    def reset(self, **kwargs) -> str:
        self.step_count = 0
        self.ambient_temp = 22.0
        self.rack_temps = [22.0, 22.0, 22.0]
        self.power_loads = [10.0, 10.0, 10.0]
        self.fan_rpms = [1000, 1000, 1000]
        self.chiller_setpoint = 15.0
        self.energy_cost = 0.15
        
        self.total_energy_consumed = 0.0
        self.broken_fans = set()
        
        self.reward = 0.0
        self.done = False
        
        return self._get_obs()
        
    def _get_obs(self) -> str:
        obs = {
            "ambient_temp": self.ambient_temp,
            "rack_temps": [round(t, 2) for t in self.rack_temps],
            "power_loads": [round(l, 2) for l in self.power_loads],
            "fan_rpms": self.fan_rpms,
            "chiller_setpoint": round(self.chiller_setpoint, 2),
            "step_count": self.step_count
        }
        return f"Observation: {json.dumps(obs)}"

    def set_fan_speed(self, rack_id: int, rpm: int) -> str:
        """
        Set the fan speed for a specific server rack.

        Args:
            rack_id: The ID of the server rack (0, 1, or 2).
            rpm: The speed to set the fan to in RPM (0 to 5000). High RPM cools faster but uses exponential power.
            
        Returns:
            Status message of the action.
        """
        if self.done: return "Episode over."
        if 0 <= rack_id < self.num_racks and rack_id not in self.broken_fans:
            self.fan_rpms[rack_id] = max(0, min(5000, rpm))
            return f"Fan {rack_id} speed bounded and set to {self.fan_rpms[rack_id]} RPM."
        return "Failed: Invalid rack_id or fan is broken."

    def adjust_chiller(self, chiller_temp: float) -> str:
        """
        Adjust the ambient liquid chiller base temperature setpoint.

        Args:
            chiller_temp: The temperature to set the chiller to (in Celsius).
            
        Returns:
            Status message.
        """
        if self.done: return "Episode over."
        self.chiller_setpoint = max(5.0, min(30.0, float(chiller_temp)))
        return f"Chiller setpoint adjusted to {self.chiller_setpoint}°C."

    def migrate_workload(self, source_rack: int, target_rack: int) -> str:
        """
        Migrate power load (heat generation) from one rack to another.

        Args:
            source_rack: The rack ID to pull workload from.
            target_rack: The rack ID to push workload to.
            
        Returns:
            Status message.
        """
        if self.done: return "Episode over."
        if 0 <= source_rack < self.num_racks and 0 <= target_rack < self.num_racks and source_rack != target_rack:
            load_to_move = self.power_loads[source_rack] * 0.5
            self.power_loads[source_rack] -= load_to_move
            self.power_loads[target_rack] += load_to_move
            return f"Migrated {load_to_move:.2f} workload from rack {source_rack} to rack {target_rack}."
        return "Failed: Invalid source or target rack."
        
    def wait(self) -> str:
        """
        Wait and progress the environment thermodynamics by 1 simulation step.

        Returns:
            The new Observation of the environment after the step.
        """
        if self.done:
            raise ValueError("Environment episode is finished.")
            
        energy_consumed = 0.0
        overheat_penalty_total = 0.0
        
        chiller_delta = max(0, self.ambient_temp - self.chiller_setpoint)
        energy_consumed += 0.5 * (chiller_delta ** 2)
        
        for i in range(self.num_racks):
            heat_generated = 0.1 * self.power_loads[i]
            rpm = self.fan_rpms[i] if i not in self.broken_fans else 0
            cooling_power = (rpm / 1000.0) * 0.5
            chiller_effect = max(0, self.rack_temps[i] - self.chiller_setpoint) * 0.1
            ambient_effect = (self.ambient_temp - self.rack_temps[i]) * 0.05
            
            self.rack_temps[i] += heat_generated - cooling_power - chiller_effect + ambient_effect
            energy_consumed += ((rpm / 1000.0) ** 3) * 0.2
            
            if self.rack_temps[i] > self.safe_temp_max:
                if self.rack_temps[i] > self.critical_temp:
                    overheat_penalty_total += self.w2_overheat
                else:
                    overheat_penalty_total += self.w2_overheat * 0.1 * (self.rack_temps[i] - self.safe_temp_max)
                    
        cost = energy_consumed * self.energy_cost
        self.total_energy_consumed += energy_consumed
        
        # Calculate step reward
        self.reward = -(self.w1_energy * cost) - overheat_penalty_total
        
        self.step_count += 1
        if self.step_count >= self.max_steps:
            self.done = True
            
        return self._get_obs()



## Define the reward function
Pinned **TRL 0.18.2** calls custom rewards as `reward_func(prompts, completions, **kwargs)` — there is no `environment_factory`. We parse `set_fan_speed`, `adjust_chiller`, `migrate_workload`, and `wait()` from each completion, step `ThermalOpsEnv`, and return the **sum of per-step rewards** after each `wait()` (same physics as the env).


In [6]:
import re


def _completion_to_text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        return "\n".join(
            str(m.get("content", "")) for m in completion if isinstance(m, dict)
        )
    return str(completion)


def _simulate_thermal_ops_from_text(text: str) -> float:
    """Run ThermalOpsEnv by parsing tool calls from free-form model output (TRL 0.18.x has no environment_factory)."""
    env = ThermalOpsEnv()
    env.reset()
    total_return = 0.0
    pos = 0
    max_ops = 200
    ops = 0
    while ops < max_ops and not env.done:
        if pos >= len(text):
            break
        chunk = text[pos:]
        best = None
        candidates = [
            (
                "wait",
                re.compile(r"\bwait\s*\(\s*\)", re.IGNORECASE).search(chunk),
            ),
            (
                "set_fan",
                re.compile(
                    r"set_fan_speed\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)", re.IGNORECASE
                ).search(chunk),
            ),
            (
                "chiller",
                re.compile(
                    r"adjust_chiller\s*\(\s*([-+]?\d*\.?\d+)\s*\)", re.IGNORECASE
                ).search(chunk),
            ),
            (
                "migrate",
                re.compile(
                    r"migrate_workload\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)", re.IGNORECASE
                ).search(chunk),
            ),
        ]
        for name, m in candidates:
            if m is None:
                continue
            abs_start = pos + m.start()
            if best is None or abs_start < best[0]:
                best = (abs_start, name, m)
        if best is None:
            break
        _, name, m = best
        pos = best[0] + len(m.group(0))
        ops += 1
        if name == "wait":
            env.wait()
            total_return += env.reward
        elif name == "set_fan":
            env.set_fan_speed(int(m.group(1)), int(m.group(2)))
        elif name == "chiller":
            env.adjust_chiller(float(m.group(1)))
        elif name == "migrate":
            env.migrate_workload(int(m.group(1)), int(m.group(2)))
    return total_return


def reward_func(prompts, completions, **kwargs) -> list[float]:
    del prompts, kwargs
    return [_simulate_thermal_ops_from_text(_completion_to_text(c)) for c in completions]

## Create Dataset for Rollouts


In [7]:
%pip install datasets jmespath
from datasets import Dataset

dataset = Dataset.from_dict({"prompt": [[{"role": "user", "content": prompt}] for _ in range(500)]})


Note: you may need to restart the kernel to use updated packages.


## Set GRPO Config

Training uses `fp16=True` like the CUDA notebook. On some ROCm stacks, if you see NaNs or instability, set `fp16=False` in `GRPOConfig` (more VRAM, slower).


In [8]:
# Hugging Face stack for the AMD/ROCm Windows path.
# Pin to a pre-masking_utils Transformers release to avoid newer torch.distributed.tensor imports
# that some Windows ROCm wheels cannot satisfy.
%pip install -U "transformers==4.50.0" "trl==0.18.2" accelerate datasets

print("Pinned Hugging Face packages installed.")

Note: you may need to restart the kernel to use updated packages.
Pinned Hugging Face packages installed.


In [9]:
import importlib
import torch
import transformers
import trl

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("GPU (PyTorch CUDA-compatible API):", torch.cuda.is_available())

try:
    importlib.import_module("torch.distributed.distributed_c10d")
    print("torch.distributed c10d: available")
except Exception as exc:
    print(f"torch.distributed c10d: unavailable ({type(exc).__name__}: {exc})")

torch: 2.9.1+rocmsdk20260116
transformers: 4.50.0
trl: 0.18.2
GPU (PyTorch CUDA-compatible API): True
torch.distributed c10d: unavailable (ModuleNotFoundError: No module named 'torch._C._distributed_c10d'; 'torch._C' is not a package)


### Windows ROCm without `c10d`

The AMD/Windows path is sensitive to the Hugging Face package versions. This notebook pins `transformers==4.50.0` and `trl==0.18.2` because newer Transformers releases pull in `masking_utils` and `torch.distributed.tensor`, which can fail on ROCm wheels that do not ship the full distributed operator set.

The **GRPO model cell** below now tries the real `torch.distributed.tensor`, `torch.distributed.distributed_c10d`, and `torch.distributed.fsdp` imports first. If your Windows ROCm wheel is missing those pieces, it falls back to **minimal import stubs** so `transformers` and TRL can load.

Treat that fallback as a narrow compatibility shim only: it is intended for **single-GPU GRPO** in this notebook with `use_vllm=False`, and it does **not** provide real distributed, dtensor, tensor-parallel, or multi-GPU support.

If anything still breaks after the pinned stack and stubs, move to **Linux + ROCm**, or use a **separate venv** with PyTorch **CUDA 12.4** wheels from pytorch.org only to restore imports on NVIDIA hardware. That CUDA path is not for AMD GPU training.


In [10]:
import sys

print("Python:", sys.executable)
# Optional LAST RESORT (separate venv recommended): CUDA PyTorch for Windows — restores c10d for TRL import.
# Will replace your current torch; restart kernel after. Not for ROCm GPU training.
# %pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu124


Python: c:\Users\Home\AppData\Local\Programs\Python\Python312\python.exe


In [11]:
import importlib
import importlib.metadata as metadata
import sys
import types
from contextlib import nullcontext

import torch

EXPECTED_TRANSFORMERS = "4.50.0"
EXPECTED_TRL = "0.18.2"

loaded_transformers = sys.modules.get("transformers")
if loaded_transformers is not None and getattr(loaded_transformers, "__version__", None) != EXPECTED_TRANSFORMERS:
    raise RuntimeError(
        "This kernel still has transformers={} loaded. Restart the kernel after the install cell so the pinned AMD-compatible stack is used.".format(
            getattr(loaded_transformers, "__version__", "unknown")
        )
    )

loaded_trl = sys.modules.get("trl")
if loaded_trl is not None and getattr(loaded_trl, "__version__", None) != EXPECTED_TRL:
    raise RuntimeError(
        "This kernel still has trl={} loaded. Restart the kernel after the install cell so the pinned AMD-compatible stack is used.".format(
            getattr(loaded_trl, "__version__", "unknown")
        )
    )

installed_transformers = metadata.version("transformers")
installed_trl = metadata.version("trl")
if installed_transformers != EXPECTED_TRANSFORMERS or installed_trl != EXPECTED_TRL:
    raise RuntimeError(
        "This notebook expects transformers=={} and trl=={} on Windows ROCm. Re-run the install cell above, then restart the kernel. Found transformers={} and trl={} instead.".format(
            EXPECTED_TRANSFORMERS,
            EXPECTED_TRL,
            installed_transformers,
            installed_trl,
        )
    )


def _install_trl_distributed_stubs_if_needed() -> None:
    """Install the smallest TRL/Transformers import-time stubs needed on Windows ROCm.

    Some Windows ROCm wheels expose GPU execution but still lack the compiled
    distributed pieces that TRL and Transformers import eagerly. These stubs
    only unblock the single-GPU `use_vllm=False` path in this notebook.
    """
    stubbed_any = False
    distributed_pkg = importlib.import_module("torch.distributed")

    tensor_pkg = "torch.distributed.tensor"
    try:
        importlib.import_module(tensor_pkg)
    except Exception:
        stubbed_any = True
        sys.modules.pop(tensor_pkg, None)
        tensor_module = types.ModuleType("tensor")
        tensor_module.__all__ = []
        sys.modules[tensor_pkg] = tensor_module
        setattr(distributed_pkg, "tensor", tensor_module)

    c10d_py = "torch.distributed.distributed_c10d"
    try:
        importlib.import_module(c10d_py)
    except Exception:
        stubbed_any = True
        sys.modules.pop(c10d_py, None)
        c10d_module = types.ModuleType("distributed_c10d")

        class ProcessGroupXCCL:
            class Options:
                pass

        def PrefixStore(prefix, store):
            return store

        c10d_module.ProcessGroupXCCL = ProcessGroupXCCL
        c10d_module.PrefixStore = PrefixStore
        sys.modules[c10d_py] = c10d_module

    fsdp_pkg = "torch.distributed.fsdp"
    try:
        importlib.import_module(fsdp_pkg)
    except Exception:
        stubbed_any = True
        for key in list(sys.modules):
            if key == fsdp_pkg or key.startswith(fsdp_pkg + "."):
                del sys.modules[key]

        class FullyShardedDataParallel:
            @staticmethod
            def summon_full_params(module, recurse=False):
                return nullcontext()

        class FSDPModule:
            pass

        def _fully_shard_unsupported(*_args, **_kwargs):
            raise RuntimeError("FSDP2/fully_shard is unavailable with the ROCm stub path.")

        fully = types.ModuleType("fully_sharded_data_parallel")
        fully.FullyShardedDataParallel = FullyShardedDataParallel
        sys.modules[fsdp_pkg + ".fully_sharded_data_parallel"] = fully

        fsdp_module = types.ModuleType("fsdp")
        fsdp_module.FullyShardedDataParallel = FullyShardedDataParallel
        fsdp_module.FSDPModule = FSDPModule
        fsdp_module.fully_sharded_data_parallel = fully
        fsdp_module.MixedPrecisionPolicy = type("MixedPrecisionPolicy", (), {})
        fsdp_module.fully_shard = _fully_shard_unsupported
        sys.modules[fsdp_pkg] = fsdp_module
        setattr(distributed_pkg, "fsdp", fsdp_module)

    if stubbed_any:
        print(
            "Note: using torch.distributed import stubs. This notebook is limited to single-GPU GRPO with use_vllm=False and no distributed/tensor-parallel features."
        )


_install_trl_distributed_stubs_if_needed()

from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
output_dir = "thermal-ops-0.5B"

# Load the model in FP32 first; this is the most stable default on Windows ROCm.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    optim="adamw_torch",
    fp16=False,
    bf16=False,
    num_generations=2,
    max_completion_length=256,
    gradient_checkpointing=True,
    use_vllm=False,
    output_dir=output_dir,
    report_to="trackio",
    push_to_hub=True,
    logging_steps=1,
    save_steps=10,
    save_total_limit=1,
)

Note: using torch.distributed import stubs. This notebook is limited to single-GPU GRPO with use_vllm=False and no distributed/tensor-parallel features.


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


## Create Trainer


In [12]:
required_symbols = [
    "model",
    "tokenizer",
    "reward_func",
    "dataset",
    "grpo_config",
    "ThermalOpsEnv",
]
missing_symbols = [name for name in required_symbols if name not in globals()]
if missing_symbols:
    raise RuntimeError(
        "Missing notebook state after restart: {}. Re-run the environment, reward function, dataset, and GRPO config cells before creating the trainer.".format(
            ", ".join(missing_symbols)
        )
    )

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
)

TypeError: GRPOTrainer.__init__() got an unexpected keyword argument 'environment_factory'

In [ ]:
print("jmespath is installed in the dataset setup cell above.")

## Memory Stats & Training Loop


In [ ]:
trainer_stats = trainer.train()


## Evaluate Inference
Run the fine-tuned model against a generic Thermal-Ops environment test.


In [ ]:
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import re

# Must match GRPOConfig.output_dir; resolved vs. notebook cwd (e.g. Colab /content).
output_dir = "thermal-ops-0.5B"
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
# If you pushed with push_to_hub, set the full repo id here (e.g. "username/thermal-ops-0.5B").
hub_model_id = None

checkpoint = Path(output_dir).resolve()
has_local = (checkpoint / "config.json").is_file()

if hub_model_id:
    load_id = hub_model_id
    fine_tuned_model = AutoModelForCausalLM.from_pretrained(
        load_id,
        torch_dtype="float32",
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(load_id)
elif has_local:
    load_id = str(checkpoint)
    fine_tuned_model = AutoModelForCausalLM.from_pretrained(
        load_id,
        torch_dtype="float32",
        device_map="auto",
        local_files_only=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(load_id, local_files_only=True)
else:
    print(
        f"No local checkpoint at {checkpoint} (no config.json). "
        f"Run training in this session or set hub_model_id to your Hub repo. "
        f"Loading base model {base_model_id} for demo."
    )
    fine_tuned_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype="float32",
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def play_thermal_ops(model, tokenizer):
    env = ThermalOpsEnv()
    
    # Introduce random difficulty (Hardware Failure Scenario)
    env.broken_fans.add(2)
    obs = env.reset()

    print("Initial observation:")
    print(obs)
    print()

    messages = [{"role": "user", "content": prompt}, {"role": "user", "content": obs}]

    for turn in range(env.max_steps * 5): # Maximum 5 tool calls per real step
        if env.done:
            break

        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=256)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        print(f"Model output: {generated_text}")
        
        try:
            # Parse tool
            if "{" in generated_text and "}" in generated_text:
                start = generated_text.index("{")
                end = generated_text.rindex("}") + 1
                args = json.loads(generated_text[start:end])
                
                tool_name = "wait"
                if "set_fan_speed" in generated_text: tool_name = "set_fan_speed"
                elif "adjust_chiller" in generated_text: tool_name = "adjust_chiller"
                elif "migrate_workload" in generated_text: tool_name = "migrate_workload"

                # Execute
                if tool_name == "set_fan_speed":
                    feedback = env.set_fan_speed(args.get("rack_id", 0), args.get("rpm", 1000))
                elif tool_name == "adjust_chiller":
                    feedback = env.adjust_chiller(args.get("chiller_temp", 20.0))
                elif tool_name == "migrate_workload":
                    feedback = env.migrate_workload(args.get("source_rack", 0), args.get("target_rack", 1))
                else:
                    feedback = env.wait()
                    
                print(f"    Feedback: {feedback.strip()}")
                
                messages.append({"role": "assistant", "content": generated_text})
                messages.append({"role": "user", "content": feedback})
            else:
                feedback = env.wait()
                messages.append({"role": "assistant", "content": generated_text})
                messages.append({"role": "user", "content": "Format error. Skipped: " + feedback})
                
        except Exception as e:
            print(f"Error executing Action: {e}")
            break

    print(f"Game finished! Final reward computed sum: {env.reward}")

play_thermal_ops(fine_tuned_model, tokenizer)

